In [8]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, ReLU, Add, MaxPooling2D, Flatten, Dense, Input
)

# ==========================================
# LOAD AUGMENTED CSV
# ==========================================
name = "sasch"
df = pd.read_csv(
    fr"C:\Users\{name}\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"
)

# ==========================================
# AUGMENTED DATASET DIRECTORY
# ==========================================

train_dir = fr"C:\Users\sasch\OneDrive\Desktop\iivp-2026-challenge\train_augmented"

X = []
y = []

# ==========================================
# LOAD IMAGES
# ==========================================

for _, row in df.iterrows():

    img_path = os.path.join(
        train_dir,
        str(row["Category"]),
        str(row["Id"]) + ".png"
    )

    img = Image.open(img_path).convert("L")

    # normalize
    img = np.array(img) / 255.0

    X.append(img)
    y.append(row["Category"])

# ==========================================
# CONVERT TO NUMPY
# ==========================================

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# ==========================================
# RESHAPE FOR ResNet-Architecture
# ==========================================

X = X.reshape(-1, 32, 32, 1)

print("CNN shape:", X.shape)

# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================
def residual_block(x, filters):
    shortcut = x

    x = Conv2D(filters, (3,3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, (3,3), padding="same")(x)
    x = BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1,1), padding="same")(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x,shortcut])
    x = ReLU()(x)

    return x

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================
def build_model():
    inputs = Input(shape=(32,32,1))

    x = Conv2D(32,(3,3), padding="same")(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = residual_block(x, 32)
    x = residual_block(x, 32)

    x = MaxPooling2D()(x)

    x = Conv2D(64,(3,3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)

    x = MaxPooling2D()(x)
    x = Flatten()(x)
    x = Dense(128, activation="relu")(x)
    outputs = Dense(10, activation="softmax")(x)

    model = Model(inputs, outputs)

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    return model
# ==========================================
# TRAIN
# ==========================================

models = []
histories = []

for i in range(3):
    print(f"Training model {i+1} of {3}")

    np.random.seed(42+i)
    tf.random.set_seed(42+i)

    model = build_model()

    history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
        verbose=1
    )

    models.append(model)
    histories.append(history)

# ==========================================
# VALIDATION ACCURACY
# ==========================================

val_loss, val_acc = model.evaluate(X_val, y_val)

print("Validation Accuracy:", val_acc)

# ==========================================
# LOAD TEST SET
# ==========================================

X_test = []
names = []

test_dir = fr"C:\Users\{name}\OneDrive\Desktop\iivp-2026-challenge\test\test"

for file in sorted(os.listdir(test_dir)):

    img_path = os.path.join(test_dir, file)

    img = Image.open(img_path).convert("L")

    img = np.array(img) / 255.0

    X_test.append(img)
    names.append(file)

X_test = np.array(X_test)

# reshape for CNN
X_test = X_test.reshape(-1, 32, 32, 1)

# ==========================================
# PREDICTIONS With Test-Time Augmentation
# ==========================================
datagen = ImageDataGenerator(
    rotation_range=12,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1
)

final_preds = []

for model in models:
    tta_preds = []

    for _ in range(10):
        aug_iter = datagen.flow(X_test, shuffle=False, batch_size=len(X_test))
        X_aug = next(aug_iter)
        tta_preds.append(model.predict(X_aug))
    final_preds.append(np.mean(tta_preds, axis=0))

y_pred = np.mean(final_preds, axis=0)
pred_labels = np.argmax(y_pred, axis=1)

# ==========================================
# SUBMISSION CSV
# ==========================================
print(len(names), len(pred_labels))

submission = pd.DataFrame({
    "Id": names,
    "Category": pred_labels
})

submission["Id"] = submission["Id"].str.replace(
    ".png",
    "",
    regex=False
)

submission.to_csv("submission_cnn.csv", index=False)

print("submission_cnn.csv saved")


Dataset shape: (34000, 32, 32)
CNN shape: (34000, 32, 32, 1)
Training model 1 of 3
Epoch 1/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 117s 132ms/step - accuracy: 0.9223 - loss: 0.3505 - val_accuracy: 0.9460 - val_loss: 0.1694
Epoch 2/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 113s 133ms/step - accuracy: 0.9839 - loss: 0.0515 - val_accuracy: 0.9428 - val_loss: 0.2519
Epoch 3/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 107s 125ms/step - accuracy: 0.9884 - loss: 0.0362 - val_accuracy: 0.9694 - val_loss: 0.1092
Epoch 4/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 102s 120ms/step - accuracy: 0.9924 - loss: 0.0250 - val_accuracy: 0.9863 - val_loss: 0.0576
Epoch 5/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 108s 127ms/step - accuracy: 0.9915 - loss: 0.0278 - val_accuracy: 0.9900 - val_loss: 0.0451
Epoch 6/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 109s 129ms/step - accuracy: 0.9947 - loss: 0.0187 - val_accuracy: 0.9853 - val_loss: 0.0600
Epoch 7/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 111s 131ms/step - accuracy: 0.9960 - loss: 0.0140 - val_accuracy: 0.9946 - val_loss: 